In [80]:
import pickle
import itertools
from pathlib import Path
import pandas as pd
import xarray as xr
import numpy as np


In [96]:
# ------------ Paths ------------
runs_dir = Path("./runs")
ensemble_metrics_dir=Path("./ensemble_metrics")
ensemble_metrics_dir.mkdir(exist_ok=True)

In [82]:
precip_products = [
    "prcp_mm_day",
    "prcp_chirps_mm_day",
    "prcp_mswep_mm_day",
    "prcp_gauge_mm_day",
]

In [83]:
all_combinations = [
    combo
    for r in range(1, len(precip_products) + 1)
    for combo in itertools.combinations(precip_products, r)
]

all_combinations

[('prcp_mm_day',),
 ('prcp_chirps_mm_day',),
 ('prcp_mswep_mm_day',),
 ('prcp_gauge_mm_day',),
 ('prcp_mm_day', 'prcp_chirps_mm_day'),
 ('prcp_mm_day', 'prcp_mswep_mm_day'),
 ('prcp_mm_day', 'prcp_gauge_mm_day'),
 ('prcp_chirps_mm_day', 'prcp_mswep_mm_day'),
 ('prcp_chirps_mm_day', 'prcp_gauge_mm_day'),
 ('prcp_mswep_mm_day', 'prcp_gauge_mm_day'),
 ('prcp_mm_day', 'prcp_chirps_mm_day', 'prcp_mswep_mm_day'),
 ('prcp_mm_day', 'prcp_chirps_mm_day', 'prcp_gauge_mm_day'),
 ('prcp_mm_day', 'prcp_mswep_mm_day', 'prcp_gauge_mm_day'),
 ('prcp_chirps_mm_day', 'prcp_mswep_mm_day', 'prcp_gauge_mm_day'),
 ('prcp_mm_day',
  'prcp_chirps_mm_day',
  'prcp_mswep_mm_day',
  'prcp_gauge_mm_day')]

In [101]:
# precip_name=all_combinations[14]
# precip_name = "_".join(precip_name)
# precip_name

In [102]:
# runs_list=list(runs_dir.glob(f"precip_{precip_name}_seed*"))
# runs_list

In [103]:
# runs=runs_list[0]
# runs

In [104]:
# pickle_path = runs / "validation" / "model_epoch030" / "validation_results.p"

# with open(pickle_path, "rb") as f:
#     data = pickle.load(f)

# data

In [105]:
# all_seeds_dfs = []

# for run in runs_list:

#     seed = run.name.split("_seed_")[1].split("_")[0]
#     pickle_path = run / "validation" / "model_epoch030" / "validation_results.p"

#     with open(pickle_path, "rb") as f:
#         data = pickle.load(f)

#     basin_dfs = []

#     for basin in data.keys():

#         ds = data[basin]['1D']['xr']
#         df = ds.to_dataframe().reset_index()
#         df = df[df["time_step"] == 0]

#         df["basin"] = basin

#         df = df.rename(columns={
#             "QObs_mm_d_sim": f"Q_sim_seed_{seed}"
#         })

#         df = df[["date", "basin", "QObs_mm_d_obs", f"Q_sim_seed_{seed}"]]

#         basin_dfs.append(df)

#     # concatenate all basins for this seed
#     seed_df = pd.concat(basin_dfs, ignore_index=True)

#     all_seeds_dfs.append(seed_df)

# # Start with first seed
# results_df = all_seeds_dfs[0]

# # Merge remaining seeds
# for df in all_seeds_dfs[1:]:
#     sim_col = [col for col in df.columns if "Q_sim_seed" in col][0]

#     results_df = results_df.merge(
#         df[["date", "basin", sim_col]],
#         on=["date", "basin"],
#         how="left"
#     )

# results_df.head()

In [100]:
# sim_cols = [col for col in results_df.columns if "Q_sim_seed" in col]
# results_df["Q_sim_median"] = results_df[sim_cols].median(axis=1)
# results_df.head()

In [ ]:
def nse(obs: xr.DataArray, sim: xr.DataArray) -> float:
    obs, sim = xr.align(obs, sim)

    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs.where(mask, drop=True)
    sim = sim.where(mask, drop=True)

    denominator = ((obs - obs.mean())**2).sum()
    numerator = ((sim - obs)**2).sum()

    return float(1 - numerator / denominator)


def mse(obs: xr.DataArray, sim: xr.DataArray) -> float:
    """Calculate mean squared error."""
    obs, sim = xr.align(obs, sim)

    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs.where(mask, drop=True)
    sim = sim.where(mask, drop=True)

    return float(((sim - obs)**2).mean())


def rmse(obs: xr.DataArray, sim: xr.DataArray) -> float:
    """Calculate root mean squared error."""
    return np.sqrt(mse(obs, sim))


def beta_kge(obs: xr.DataArray, sim: xr.DataArray) -> float:
    """Calculate the beta KGE term (ratio of means)."""
    obs, sim = xr.align(obs, sim)

    mask = np.isfinite(obs) & np.isfinite(sim)
    obs = obs.where(mask, drop=True)
    sim = sim.where(mask, drop=True)

    return float(sim.mean() / obs.mean())

In [98]:
# metrics_data = []

# for basin, df_basin in results_df.groupby("basin"):

#     obs = xr.DataArray(
#         df_basin["QObs_mm_d_obs"].values,
#         dims=["time"],
#         coords={"time": df_basin["date"].values}
#     )

#     sim = xr.DataArray(
#         df_basin["Q_sim_median"].values,
#         dims=["time"],
#         coords={"time": df_basin["date"].values}
#     )

#     metrics_data.append({
#         "basin": basin,
#         "NSE": nse(obs, sim),
#         "RMSE": rmse(obs, sim),
#         "MSE": mse(obs, sim),
#         "Beta-KGE": beta_kge(obs, sim)
#     })

# metrics_df = pd.DataFrame(metrics_data)
# metrics_df.to_csv(ensemble_metrics_dir / f"{precip_name}.csv", index=False)

In [99]:
for combo in all_combinations:
    precip_name = "_".join(combo)
    print(f"Processing: {precip_name}")
    
    runs_list = list(runs_dir.glob(f"precip_{precip_name}_seed*"))
    
    if not runs_list:
        print(f"  No runs found, skipping...")
        continue
    
    all_seeds_dfs = []
    
    for run in runs_list:
        seed = run.name.split("_seed_")[1].split("_")[0]
        pickle_path = run / "validation" / "model_epoch030" / "validation_results.p"
        
        if not pickle_path.exists():
            continue
            
        with open(pickle_path, "rb") as f:
            data = pickle.load(f)
        
        basin_dfs = []
        
        for basin in data.keys():
            ds = data[basin]['1D']['xr']
            df = ds.to_dataframe().reset_index()
            df = df[df["time_step"] == 0]
            df["basin"] = basin
            df = df.rename(columns={"QObs_mm_d_sim": f"Q_sim_seed_{seed}"})
            df = df[["date", "basin", "QObs_mm_d_obs", f"Q_sim_seed_{seed}"]]
            basin_dfs.append(df)
        
        seed_df = pd.concat(basin_dfs, ignore_index=True)
        all_seeds_dfs.append(seed_df)
    
    if not all_seeds_dfs:
        print(f"  No valid data, skipping...")
        continue
    
    # Start with first seed
    results_df = all_seeds_dfs[0]
    
    # Merge remaining seeds
    for df in all_seeds_dfs[1:]:
        sim_col = [col for col in df.columns if "Q_sim_seed" in col][0]
        results_df = results_df.merge(
            df[["date", "basin", sim_col]],
            on=["date", "basin"],
            how="left"
        )
    
    # Calculate median
    sim_cols = [col for col in results_df.columns if "Q_sim_seed" in col]
    results_df["Q_sim_median"] = results_df[sim_cols].median(axis=1)
    
    # Compute metrics for each basin
    metrics_data = []
    
    for basin, df_basin in results_df.groupby("basin"):
        obs = xr.DataArray(
            df_basin["QObs_mm_d_obs"].values,
            dims=["time"],
            coords={"time": df_basin["date"].values}
        )
        sim = xr.DataArray(
            df_basin["Q_sim_median"].values,
            dims=["time"],
            coords={"time": df_basin["date"].values}
        )
        
        metrics_data.append({
            "basin": basin,
            "NSE": nse(obs, sim),
            "RMSE": rmse(obs, sim),
            "MSE": mse(obs, sim),
            "Beta-KGE": beta_kge(obs, sim)
        })
    
    metrics_df = pd.DataFrame(metrics_data)
    metrics_df.to_csv(ensemble_metrics_dir / f"{precip_name}.csv", index=False)
    print(f"  Saved: {precip_name}.csv")

print("Done!")

Processing: prcp_mm_day
  Saved: prcp_mm_day.csv
Processing: prcp_chirps_mm_day
  Saved: prcp_chirps_mm_day.csv
Processing: prcp_mswep_mm_day
  Saved: prcp_mswep_mm_day.csv
Processing: prcp_gauge_mm_day
  Saved: prcp_gauge_mm_day.csv
Processing: prcp_mm_day_prcp_chirps_mm_day
  Saved: prcp_mm_day_prcp_chirps_mm_day.csv
Processing: prcp_mm_day_prcp_mswep_mm_day
  Saved: prcp_mm_day_prcp_mswep_mm_day.csv
Processing: prcp_mm_day_prcp_gauge_mm_day
  Saved: prcp_mm_day_prcp_gauge_mm_day.csv
Processing: prcp_chirps_mm_day_prcp_mswep_mm_day
  Saved: prcp_chirps_mm_day_prcp_mswep_mm_day.csv
Processing: prcp_chirps_mm_day_prcp_gauge_mm_day
  Saved: prcp_chirps_mm_day_prcp_gauge_mm_day.csv
Processing: prcp_mswep_mm_day_prcp_gauge_mm_day
  Saved: prcp_mswep_mm_day_prcp_gauge_mm_day.csv
Processing: prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day
  Saved: prcp_mm_day_prcp_chirps_mm_day_prcp_mswep_mm_day.csv
Processing: prcp_mm_day_prcp_chirps_mm_day_prcp_gauge_mm_day
  Saved: prcp_mm_day_prcp_chir